<a href="https://colab.research.google.com/github/korzhimanov/dsp-seminars/blob/main/seminars/3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Практическое занятие №3

## Свёртка и корреляционный анализ сигналов

## Цели занятия
- Освоить вычисление свёртки и корреляции в Python.
- Применить свёртку для фильтрации сигналов.
- Использовать кросс-корреляцию для поиска временной задержки и обнаружения шаблона.
- Проанализировать влияние уровня шума на точность оценок.

## Подготовка окружения

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal
from scipy.io import wavfile
try:
    from IPython.display import Audio, display
except ImportError:
    Audio = None

    def display(obj):
        return obj
import time

%matplotlib inline
plt.rcParams['figure.figsize'] = (12, 4)

## Задание 1. Свёртка гауссовых функций: сравнение численного и аналитического результатов

### Теоретическое введение
Свёртка двух гауссовых функций даёт гауссову функцию с дисперсией, равной сумме дисперсий. Если
$$
f(t) = \frac{1}{\sqrt{2\pi\sigma_1^2}} e^{-t^2/(2\sigma_1^2)}, \quad g(t) = \frac{1}{\sqrt{2\pi\sigma_2^2}} e^{-t^2/(2\sigma_2^2)},
$$
то
$$
(f * g)(t) = \frac{1}{\sqrt{2\pi(\sigma_1^2+\sigma_2^2)}} e^{-t^2/(2(\sigma_1^2+\sigma_2^2))}.
$$

### Задание
1. Сгенерируйте дискретные гауссовы импульсы (например, с помощью `scipy.signal.windows.gaussian`) с заданными стандартными отклонениями `sigma1=3`, `sigma2=5`. Используйте длину окна, достаточную для захвата всей значимой части (например, 10*sigma).
2. Вычислите свёртку численно с помощью `np.convolve`.
3. Постройте графики:
   - Исходные функции.
   - Результат свёртки (численный).
   - Теоретическую гауссову функцию с дисперсией `sigma1^2 + sigma2^2`.
4. Оцените среднеквадратичную ошибку между численным и теоретическим результатами.

In [ ]:
sigma1, sigma2 = 3, 5
M1 = int(np.ceil(10 * sigma1)) | 1
M2 = int(np.ceil(10 * sigma2)) | 1

x1 = np.arange(M1) - (M1 - 1) / 2
x2 = np.arange(M2) - (M2 - 1) / 2

g1 = signal.windows.gaussian(M1, std=sigma1)
g2 = signal.windows.gaussian(M2, std=sigma2)
g1 /= g1.sum()
g2 /= g2.sum()

conv_num = np.convolve(g1, g2, mode='full')
x_conv = np.arange(len(conv_num)) - ((M1 - 1) / 2 + (M2 - 1) / 2)

sigma_theory = np.sqrt(sigma1 ** 2 + sigma2 ** 2)
g_theory = np.exp(-0.5 * (x_conv / sigma_theory) ** 2)
g_theory /= g_theory.sum()

rmse = np.sqrt(np.mean((conv_num - g_theory) ** 2))

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].plot(x1, g1, label=f'Gaussian 1, $\sigma$={sigma1}')
axes[0].plot(x2, g2, label=f'Gaussian 2, $\sigma$={sigma2}')
axes[0].set_title('Исходные гауссовы импульсы')
axes[0].set_xlabel('Отсчёт')
axes[0].set_ylabel('Амплитуда')
axes[0].legend()
axes[0].grid(True)

axes[1].plot(x_conv, conv_num, label='Численная свёртка', linewidth=2)
axes[1].plot(x_conv, g_theory, '--', label=f'Теория, $\sigma$={sigma_theory:.3f}')
axes[1].set_title('Свёртка: численно и аналитически')
axes[1].set_xlabel('Отсчёт')
axes[1].set_ylabel('Амплитуда')
axes[1].legend()
axes[1].grid(True)
plt.tight_layout()
plt.show()

print(f'Размеры окон: M1={M1}, M2={M2}')
print(f'Теоретическое sigma после свёртки: {sigma_theory:.6f}')
print(f'RMSE между численным и аналитическим результатами: {rmse:.2e}')


**Вопросы:**
- Что произойдёт с формой свёртки при увеличении σ?

При увеличении $\sigma$ свёртка становится шире и более гладкой: её эффективная дисперсия растёт как $\sigma_{\text{conv}} = \sqrt{\sigma_1^2 + \sigma_2^2}$. Из-за нормировки площадь под кривой сохраняется, поэтому максимум становится ниже, а сама кривая менее острой.

## Задание 2. Фильтрация с помощью свёртки: сравнение прямоугольного и гауссовского окон

### Цель
Изучить, как выбор ядра и его длина влияют на подавление высокочастотной составляющей и сохранение низкочастотной.

### Задание
1. Создайте сигнал длительностью 2 секунды, частота дискретизации 1000 Гц, состоящий из суммы двух синусоид:
   - низкая частота \( f_1 = 5 \) Гц, амплитуда 1,
   - высокая частота \( f_2 = 80 \) Гц, амплитуда 0.5.
2. Сгенерируйте прямоугольное окно длины `L` (например, `L = 21`, используйте `scipy.signal.windows.boxcar(L)`), нормализованное так, чтобы сумма коэффициентов была 1.
3. Сгенерируйте гауссовское окно той же длины (используйте `signal.windows.gaussian(L, std=L/5)`), также нормализованное.
4. Примените свёртку с этими окнами (используйте `mode='same'`).
5. Постройте графики:
   - Исходный сигнал (первые 0.5 с).
   - Отфильтрованные сигналы для обоих окон.
6. Вычислите и сравните амплитуды полезной составляющей (5 Гц) и подавленной (80 Гц) после фильтрации. Для этого:
   - Возьмите БПФ сигналов,
   - Измерьте амплитуды на соответствующих частотах.
7. Исследуйте влияние длины окна: повторите для длин `L = 11, 21, 41, 81`.

In [ ]:
fs = 1000
duration = 2.0
t = np.arange(0, duration, 1 / fs)
x = np.sin(2 * np.pi * 5 * t) + 0.5 * np.sin(2 * np.pi * 80 * t)

lengths = [11, 21, 41, 81]


def amplitude_at(signal_data, fs, target_freq):
    spectrum = np.fft.rfft(signal_data)
    freqs = np.fft.rfftfreq(len(signal_data), d=1 / fs)
    idx = np.argmin(np.abs(freqs - target_freq))
    return 2 * np.abs(spectrum[idx]) / len(signal_data)


results = []
example_signals = {}
for L in lengths:
    rect = signal.windows.boxcar(L) / L
    gauss = signal.windows.gaussian(L, std=L / 5)
    gauss /= gauss.sum()

    y_rect = np.convolve(x, rect, mode='same')
    y_gauss = np.convolve(x, gauss, mode='same')

    results.append({
        'L': L,
        'rect_5': amplitude_at(y_rect, fs, 5),
        'rect_80': amplitude_at(y_rect, fs, 80),
        'gauss_5': amplitude_at(y_gauss, fs, 5),
        'gauss_80': amplitude_at(y_gauss, fs, 80),
    })

    if L == 21:
        example_signals['rect'] = y_rect
        example_signals['gauss'] = y_gauss

plot_mask = t <= 0.5
fig, axes = plt.subplots(1, 2, figsize=(15, 4))
axes[0].plot(t[plot_mask], x[plot_mask], label='Исходный сигнал', alpha=0.7)
axes[0].plot(t[plot_mask], example_signals['rect'][plot_mask], label='Прямоугольное окно, L=21')
axes[0].plot(t[plot_mask], example_signals['gauss'][plot_mask], label='Гауссово окно, L=21')
axes[0].set_title('Фильтрация во временной области')
axes[0].set_xlabel('Время, с')
axes[0].set_ylabel('Амплитуда')
axes[0].legend()
axes[0].grid(True)

axes[1].plot(lengths, [row['rect_80'] for row in results], 'o-', label='80 Гц, прямоугольное')
axes[1].plot(lengths, [row['gauss_80'] for row in results], 'o-', label='80 Гц, гауссово')
axes[1].plot(lengths, [row['rect_5'] for row in results], 's--', label='5 Гц, прямоугольное')
axes[1].plot(lengths, [row['gauss_5'] for row in results], 's--', label='5 Гц, гауссово')
axes[1].set_title('Амплитуды после фильтрации')
axes[1].set_xlabel('Длина окна L')
axes[1].set_ylabel('Оценка амплитуды')
axes[1].legend()
axes[1].grid(True)
plt.tight_layout()
plt.show()

print('L | rect A(5 Гц) | rect A(80 Гц) | gauss A(5 Гц) | gauss A(80 Гц)')
for row in results:
    print(
        f"{row['L']:>2} | {row['rect_5']:.4f} | {row['rect_80']:.4f} | "
        f"{row['gauss_5']:.4f} | {row['gauss_80']:.4f}"
    )


**Вопросы:**
- Какое окно (прямоугольное или гауссовское) даёт лучшее подавление высокой частоты при одинаковой длине?
- При какой длине прямоугольного и гауссовского окон амплитуда высокочастотного амплитуда падает в 10 раз?
- Как увеличение длины окна влияет на подавление сохранение амплитуды низкой частоты? При какой длине окон она уменьшается более чем на 10%?

По результатам расчёта для набора $L = 11, 21, 41, 81$ гауссово окно обычно лучше подавляет 80 Гц при той же длине и при этом лучше сохраняет 5 Гц. Исключение в этом эксперименте: при очень коротком окне $L=11$ прямоугольное усреднение ослабило 80 Гц сильнее.

Падение амплитуды высокочастотной составляющей примерно в 10 раз наблюдается так:
- прямоугольное окно: начиная примерно с $L=41$;
- гауссово окно: уже около $L=21$ (для $L=21$ амплитуда почти ровно $0.05$, а для $L=41$ подавление уже намного сильнее).

С ростом длины окна подавление 80 Гц усиливается, но вместе с этим начинает заметнее снижаться и полезная составляющая 5 Гц. В этом расчёте спад амплитуды 5 Гц более чем на 10% появляется только при $L=81$ для обоих окон; при этом гауссово окно сохраняет низкую частоту лучше, чем прямоугольное.

## Задание 3. Поиск временной задержки с помощью кросс-корреляции

### Цель
Определить временной сдвиг между двумя сигналами в присутствии шума и оценить влияние уровня шума на точность.

### Задание
1. Сгенерируйте сигнал длительностью 1 секунду (fs=1000 Гц), представляющий собой сумму 100 синусоид со случайными частотами (в диапазоне 10–100 Гц), амплитудами (в диапазоне 0.5–1.5 Гц) и фазами (в диапазоне 0–2$\pi$). Воспользуйтесь функцией `numpy.random.uniform`. Нормализуйте полученный сигнал в интервале $[-1;1]$. Назовём его `x`.
2. Создайте второй сигнал `y`, который является сдвинутой во времени копией `x` на `delay` отсчётов (выберите задержку, например, 100 отсчётов) и добавьте к нему случайный сигнал (шум) с уровнем `noise_level` (например, 0.1) с помощью функции `numpy.random.randn`.
3. Вычислите кросс-корреляцию `corr = np.correlate(x, y, mode='full')` и найдите индекс максимума. Оцените задержку как `delay_est = argmax(corr) - (len(x)-1)` (поскольку `mode='full'` даёт диапазон от -N+1 до N-1).
4. Повторите эксперимент для различных уровней шума в диапазоне от `0` до `2`.
5. Для двух уровней шума (0.3 и 2) постройте график кросс-корреляции и отметьте положение истинной и найденной задержки.

In [ ]:
np.random.seed(42)
fs = 1000
t = np.arange(0, 1, 1 / fs)

freqs = np.random.uniform(10, 100, size=100)
amplitudes = np.random.uniform(0.5, 1.5, size=100)
phases = np.random.uniform(0, 2 * np.pi, size=100)

x = np.sum(
    amplitudes[:, None] * np.sin(2 * np.pi * freqs[:, None] * t + phases[:, None]),
    axis=0,
)
x /= np.max(np.abs(x))

delay = 100
y_clean = np.concatenate([x[delay:], np.zeros(delay)])
noise_levels = np.linspace(0, 2, 21)
n_trials = 100


def estimate_delay(noise_level, seed):
    rng = np.random.default_rng(seed)
    y = y_clean + noise_level * rng.standard_normal(len(x))
    corr = np.correlate(x, y, mode='full')
    lags = np.arange(-len(x) + 1, len(x))
    delay_est = int(lags[np.argmax(corr)])
    return delay_est, corr, lags


stats = []
for noise_level in noise_levels:
    estimates = []
    for seed in range(n_trials):
        delay_est, _, _ = estimate_delay(noise_level, 1000 + seed)
        estimates.append(delay_est)

    estimates = np.array(estimates)
    stats.append({
        'noise': noise_level,
        'mean_est': estimates.mean(),
        'std_est': estimates.std(),
        'frac_large_error': np.mean(np.abs(estimates - delay) > 0.1 * delay),
        'frac_near_zero': np.mean(np.abs(estimates) <= 10),
    })

# Для большого шума берём реализацию, где ложный пик хорошо виден на графике.
example_seeds = {0.3: 1000, 2.0: 1036}
example_results = {noise: estimate_delay(noise, seed) for noise, seed in example_seeds.items()}

fig, axes = plt.subplots(3, 1, figsize=(12, 12))
axes[0].plot(noise_levels, [row['std_est'] for row in stats], 'o-', label='STD оценки задержки')
axes[0].plot(noise_levels, [row['frac_large_error'] for row in stats], 's-', label='Доля ошибок > 10%')
axes[0].plot(noise_levels, [row['frac_near_zero'] for row in stats], '^-', label='Доля оценок около 0')
axes[0].set_title('Устойчивость оценки задержки')
axes[0].set_xlabel('Уровень шума')
axes[0].set_ylabel('Метрика')
axes[0].legend()
axes[0].grid(True)

for ax, noise_level in zip(axes[1:], [0.3, 2.0]):
    delay_est, corr, lags = example_results[noise_level]
    ax.plot(lags, corr)
    ax.axvline(delay, color='green', linestyle='--', label=f'Истинная задержка = {delay}')
    ax.axvline(delay_est, color='red', linestyle='--', label=f'Оценка = {delay_est}')
    ax.set_title(f'Кросс-корреляция при noise_level = {noise_level}')
    ax.set_xlabel('Лаг, отсчёты')
    ax.set_ylabel('Корреляция')
    ax.legend()
    ax.grid(True)

plt.tight_layout()
plt.show()

first_bad = next((row['noise'] for row in stats if row['frac_large_error'] > 0), None)
first_near_zero = next((row['noise'] for row in stats if row['frac_near_zero'] > 0.05), None)

print('noise | mean(delay_est) | std(delay_est) | P(|err| > 10) | P(|delay_est| <= 10)')
for row in stats:
    print(
        f"{row['noise']:.1f} | {row['mean_est']:.2f} | {row['std_est']:.2f} | "
        f"{row['frac_large_error']:.2f} | {row['frac_near_zero']:.2f}"
    )

print(f'Первые редкие ошибки больше 10% появляются примерно при noise_level = {first_bad}.')
if first_near_zero is None:
    print('В диапазоне noise_level от 0 до 2 оценка не становится практически неотличимой от 0.')
else:
    print(f'Оценка становится близкой к 0 примерно при noise_level = {first_near_zero}.')


**Вопросы:**
- Как зависит точность оценки задержки от уровня шума? При каком уровне шума вычисленная задержка отличается от истинной более чем на 10%? При каком уровне шума вычисленная задержка становится практически неотличима от `0`?
- Почему при высоком уровне шума могут появляться ложные пики?

При увеличении уровня шума оценка задержки долго остаётся устойчивой: до `noise_level` порядка `1.5` она почти всегда совпадает с истинными `100` отсчётами. Единичные ошибки больше 10% начинают появляться примерно при `noise_level ≈ 1.6-1.7`, а к `noise_level = 2.0` доля таких срывов в серии прогонов становится заметной.

В этом эксперименте в диапазоне шума от `0` до `2` оценка не стала практически неотличимой от `0`: из-за богатого по спектру сигнала главный пик корреляции всё ещё чаще остаётся около истинной задержки. При больших шумах начинают появляться ложные пики, потому что случайные выбросы шума местами дают частичное совпадение формы сигналов и создают локальные максимумы, сравнимые с истинным пиком.

## Задание 4. Обнаружение шаблона в зашумлённом сигнале

### Цель
Найти местоположение сигнала, имеющего форму гауссова импульса, модулированного синусоидой, в смеси с шумом. Исследовать влияние отношения сигнал/шум на точность обнаружения.

### Задание
1. Создайте шаблон `template` – произведение гауссовой огибающей (σ=100 отсчётов) на синусоиду частотой 20 Гц (при fs=1000 Гц). Длина шаблона примерно 6σ.
2. Создайте длинный сигнал `long_signal` длиной 2000 отсчётов, состоящий из:
   - случайного сигнала (шума), созданного с помощью `numpy.random.randn()` с единичной амплитудой,
   - вставленного в случайную позицию шаблона, умноженного на амплитуду `A` (например, 2).
3. Используйте кросс-корреляцию `np.correlate(long_signal, template, mode='valid')` для поиска позиции шаблона. Найдите индекс максимума корреляции.
4. Повторите эксперимент для разных отношений сигнал/шум (SNR), варьируя амплитуду шаблона от 0.2 до 5.
5. Для одного значения SNR (например, 1) визуализируйте: исходный длинный сигнал, шаблон, результат кросс-корреляции с отмеченным пиком.

In [ ]:
np.random.seed(42)
fs = 1000
sigma = 100
n = np.arange(-3 * sigma, 3 * sigma)
template = np.exp(-0.5 * (n / sigma) ** 2) * np.sin(2 * np.pi * 20 * n / fs)
template_rms = np.sqrt(np.mean(template ** 2))

long_len = 2000
amps = np.linspace(0.2, 5.0, 25)
n_trials = 20
metrics = []

for A in amps:
    errors = []
    hits = 0
    for seed in range(n_trials):
        rng = np.random.default_rng(5000 + seed)
        long_signal = rng.standard_normal(long_len)
        insert_pos = int(rng.integers(0, long_len - len(template) + 1))
        long_signal[insert_pos:insert_pos + len(template)] += A * template

        corr = np.correlate(long_signal, template, mode='valid')
        estimate = int(np.argmax(corr))
        error = abs(estimate - insert_pos)
        errors.append(error)
        hits += int(error == 0)

    snr_linear = A * template_rms
    metrics.append({
        'A': A,
        'snr_db': 20 * np.log10(snr_linear),
        'hit_rate': hits / n_trials,
        'mean_abs_error': np.mean(errors),
    })

example_A = 1.0
rng = np.random.default_rng(2024)
long_signal = rng.standard_normal(long_len)
insert_pos = int(rng.integers(0, long_len - len(template) + 1))
long_signal[insert_pos:insert_pos + len(template)] += example_A * template
corr = np.correlate(long_signal, template, mode='valid')
estimate = int(np.argmax(corr))

fig, axes = plt.subplots(2, 2, figsize=(14, 8))
axes[0, 0].plot(long_signal)
axes[0, 0].axvline(insert_pos, color='green', linestyle='--', label='Истинная позиция')
axes[0, 0].axvline(estimate, color='red', linestyle='--', label='Оценка')
axes[0, 0].set_title(f'Длинный сигнал, A = {example_A}')
axes[0, 0].set_xlabel('Отсчёт')
axes[0, 0].set_ylabel('Амплитуда')
axes[0, 0].legend()
axes[0, 0].grid(True)

axes[0, 1].plot(template)
axes[0, 1].set_title('Шаблон')
axes[0, 1].set_xlabel('Отсчёт')
axes[0, 1].set_ylabel('Амплитуда')
axes[0, 1].grid(True)

axes[1, 0].plot(corr)
axes[1, 0].axvline(insert_pos, color='green', linestyle='--', label='Истинная позиция')
axes[1, 0].axvline(estimate, color='red', linestyle='--', label='Оценка')
axes[1, 0].set_title('Кросс-корреляция')
axes[1, 0].set_xlabel('Позиция')
axes[1, 0].set_ylabel('Корреляция')
axes[1, 0].legend()
axes[1, 0].grid(True)

axes[1, 1].plot([row['snr_db'] for row in metrics], [row['hit_rate'] for row in metrics], 'o-', label='Точность обнаружения')
axes[1, 1].plot([row['snr_db'] for row in metrics], [row['mean_abs_error'] for row in metrics], 's--', label='Средняя ошибка')
axes[1, 1].set_title('Качество обнаружения vs SNR')
axes[1, 1].set_xlabel('SNR, дБ')
axes[1, 1].set_ylabel('Метрика')
axes[1, 1].legend()
axes[1, 1].grid(True)

plt.tight_layout()
plt.show()

print('A | SNR, dB | hit rate | mean |error|')
for row in metrics:
    print(f"{row['A']:.1f} | {row['snr_db']:.2f} | {row['hit_rate']:.2f} | {row['mean_abs_error']:.2f}")


**Вопросы:**
- При каком SNR результаты измерения резко ухудшаются?

Результаты резко ухудшаются примерно ниже `SNR ≈ -10 дБ`, что в этом эксперименте соответствует амплитуде шаблона около `A = 0.8-1.0` и меньше. Выше этого уровня пик корреляции обычно остаётся доминирующим, а ниже него шум начинает часто создавать конкурирующие максимумы, поэтому точность быстро падает.

## Задание 5. Поиск фрагмента в реальном аудиосигнале

### Цель
Применить кросс-корреляцию для нахождения заданного фрагмента в аудиофайле. Фрагмент и основной файл предоставлены (студентам нужно будет загрузить их).

### Задание
1. Загрузите аудиофайл `full_audio.wav` и фрагмент `fragment.wav`. Используйте `scipy.io.wavfile.read`.
2. Вычислите кросс-корреляцию между полным сигналом и фрагментом.
3. Найдите позицию максимального значения корреляции и определите временное смещение (в отсчётах и в секундах).
4. Постройте график кросс-корреляции и отметьте найденный пик.
5. Вырежьте из полного сигнала участок, соответствующий найденной позиции, и прослушайте его (используйте `Audio`). Убедитесь, что он совпадает с фрагментом.

In [ ]:
from pathlib import Path

candidate_dirs = [Path('data'), Path('../data')]
full_audio_path = None
fragment_path = None

for directory in candidate_dirs:
    full_candidate = directory / 'full_audio.wav'
    fragment_candidate = directory / 'fragment.wav'
    if full_candidate.exists() and fragment_candidate.exists():
        full_audio_path = full_candidate.resolve()
        fragment_path = fragment_candidate.resolve()
        break

if full_audio_path is None or fragment_path is None:
    raise FileNotFoundError('Не найдены full_audio.wav и fragment.wav ни в data/, ни в ../data/.')

print(f'full_audio_path = {full_audio_path}')
print(f'fragment_path = {fragment_path}')


In [ ]:
def to_mono_float(data):
    data = data.astype(np.float64)
    if data.ndim == 2:
        data = data.mean(axis=1)
    data -= data.mean()
    peak = np.max(np.abs(data))
    if peak > 0:
        data /= peak
    return data


rate_full, full_audio = wavfile.read(full_audio_path)
rate_frag, fragment = wavfile.read(fragment_path)
if rate_full != rate_frag:
    raise ValueError('Частоты дискретизации должны совпадать.')

full_audio = to_mono_float(full_audio)
fragment = to_mono_float(fragment)

corr = np.correlate(full_audio, fragment, mode='valid')
peak_idx = int(np.argmax(corr))
peak_time = peak_idx / rate_full
matched_fragment = full_audio[peak_idx:peak_idx + len(fragment)].copy()
matched_fragment /= np.max(np.abs(matched_fragment))

match_corr = np.corrcoef(matched_fragment, fragment)[0, 1]
peaks, props = signal.find_peaks(corr, distance=len(fragment) // 2, height=0.3 * corr.max())

corr_time = np.arange(len(corr)) / rate_full
fragment_time = np.arange(len(fragment)) / rate_frag

fig, axes = plt.subplots(2, 1, figsize=(12, 8))
axes[0].plot(corr_time, corr)
axes[0].axvline(peak_time, color='red', linestyle='--', label=f'Пик: {peak_time:.3f} с')
axes[0].set_title('Кросс-корреляция полного сигнала и фрагмента')
axes[0].set_xlabel('Время, с')
axes[0].set_ylabel('Корреляция')
axes[0].legend()
axes[0].grid(True)

axes[1].plot(fragment_time, fragment, label='Фрагмент')
axes[1].plot(fragment_time, matched_fragment, '--', label='Найденный участок')
axes[1].set_title('Сравнение фрагмента и найденного участка')
axes[1].set_xlabel('Время, с')
axes[1].set_ylabel('Амплитуда')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.show()

print(f'Смещение: {peak_idx} отсчётов ({peak_time:.6f} с)')
print(f'Коэффициент корреляции между фрагментом и найденным участком: {match_corr:.6f}')
print(f'Количество сильных пиков (>= 30% от максимума): {len(peaks)}')
print('Времена сильных пиков, с:', [round(float(p / rate_full), 3) for p in peaks])

if Audio is not None:
    display(Audio(fragment, rate=rate_frag))
    display(Audio(matched_fragment, rate=rate_full))
else:
    print('IPython.display недоступен, поэтому воспроизведение аудио пропущено.')


**Вопросы:**
- Почему перед вычислением корреляции сигналы следует нормализовать?
- Сколько раз в сигнале встречается вырезанное во фрагменте слово? Удаётся ли с помощью корреляционного анализа определить его во всех случаях? Возникают ли ложные максимумы?
- Что будет, если фрагмент не содержится в полном сигнале? Как это отразится на корреляции?

Перед корреляцией сигналы полезно нормализовать, чтобы максимум определялся сходством формы, а не просто большей амплитудой одного из сигналов. Дополнительно вычитание среднего убирает постоянную составляющую, которая иначе может искусственно увеличивать корреляцию.

В этом аудиофайле фрагмент даёт один отчётливый сильный пик: найдено одно уверенное совпадение на позиции `279235` отсчётов, то есть примерно `6.332` с. Для него совпадение почти идеальное: корреляция между вырезанным участком и исходным фрагментом близка к `1`. Ложные максимумы есть, но они намного слабее главного пика.

Если бы фрагмент не содержался в полном сигнале, кросс-корреляция не имела бы одного узкого доминирующего максимума: максимум стал бы сопоставим с фоном, а решение пришлось бы принимать по порогу или дополнительным признакам.